# Module 5 — LangGraph: From For-Loop to State Machine

**Course:** Build Your First AI Agent from Scratch  
**Community:** [Autonomous MSP](https://skool.com/autonomous-msp-2162)

**What you build in this module:**
- `AgentState` — a TypedDict that holds everything the agent knows
- 4 nodes: `observe_node`, `reason_node`, `act_node`, `verify_node`
- `route_after_reason()` — the routing function that controls the loop
- `build_agent()` — wires all nodes into a compiled LangGraph

**The networking analogy:** A for-loop is hoping traffic finds its way. LangGraph is policy-based routing for your agent's logic. Same input → same path, every time.

## Step 1 — Install Dependencies

In [ ]:
!pip install langgraph langchain-anthropic chromadb -q

## Step 2 — Set Your API Key

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # replace with your key
    print("API key set directly.")

## Step 3 — Why Replace the For-Loop?

The Module 2 for-loop works. But it has four hard limitations in production:

1. **Cannot branch** — every problem follows the same path. No way to say "BGP → left, OSPF → right"
2. **Cannot pause** — if you need human approval mid-loop, you can't pause and resume
3. **Cannot inspect state** — when something goes wrong, you don't know where the agent was
4. **Cannot retry a specific step** — if iteration 3 produces garbage, you restart from scratch

LangGraph fixes all four.

In [ ]:
# Visual comparison: For-loop vs LangGraph

print("=== For-Loop (Module 2) ===")
print("""
for i in range(max_iterations):
    response = llm.call(context)
    if 'DONE' in response:
        break
    execute_tool(response)

Problems:
  - No named states (where is the agent right now?)
  - No conditional routing (BGP vs OSPF same path)
  - No pause/resume for human approval
  - No checkpointing for debugging
""")

print("=== LangGraph (This Module) ===")
print("""
OBSERVE → REASON → ACT → REASON → ... → VERIFY → END
                 ↑___________if confidence < 0.85___|

Benefits:
  + Named states: always know what the agent is doing
  + Conditional routing: different paths for different problems
  + Pause on human approval, resume from exact state
  + Full checkpointing: replay any step for debugging
""")

## Step 4 — AgentState: The Single Source of Truth

Every node receives the current state, does its work, and returns only the fields that changed. LangGraph merges those changes into the full state before calling the next node.

**Why TypedDict instead of a class?** LangGraph checkpoints state to disk (or memory) after every node. Checkpointing requires JSON serialization. TypedDict is a plain dict under the hood — LangGraph knows how to handle it.

In [ ]:
from typing import TypedDict, List, Optional


class AgentState(TypedDict):
    # The original question — never modified after initialization
    user_query:     str

    # Which node is currently active: observe / reason / act / verify
    phase:          str

    # How many reason-act cycles have completed
    iteration:      int

    # Safety valve: if iteration reaches this, force verify regardless of confidence
    max_iterations: int

    # Text block from ChromaDB — past incidents relevant to this query
    rag_context:    str

    # Grows as tools run: each tool result appended as a string
    evidence:       List[str]

    # LLM's current best explanation, updated every reason_node call
    hypothesis:     str

    # 0.0 to 1.0 — when this reaches 0.85, routing sends agent to verify
    confidence:     float

    # List of every tool called so far (for the final report)
    tool_calls:     List[str]

    # The tool reason_node chose to call next — act_node executes this
    pending_tool:   Optional[str]

    # Populated by verify_node — this is what goes back to the engineer
    final_answer:   str

    # Set to True by verify_node — signals the graph to terminate
    done:           bool


print("AgentState fields:", list(AgentState.__annotations__.keys()))
print("\nKey insight: 'confidence' is what drives routing.")
print("  < 0.85 → keep collecting evidence (act_node)")
print("  >= 0.85 → move to verify_node")

## Step 5 — Demo Tools and ChromaDB Setup

These are the same demo tools from Module 2, plus a minimal ChromaDB setup. In a full production agent, these would be replaced with the real tools from Module 3 and the full ChromaDB from Module 4.

In [ ]:
import chromadb
from langchain_anthropic import ChatAnthropic

# ── LLM ───────────────────────────────────────────────────────
llm = ChatAnthropic(model="claude-sonnet-4-6")

# ── ChromaDB (in-memory for this notebook) ────────────────────
chroma = chromadb.Client()
collection = chroma.get_or_create_collection(
    name="incidents",
    metadata={"hnsw:space": "cosine"}
)

# Seed with one incident so observe_node has something to find
collection.upsert(
    ids=["INC-1955"],
    documents=["OSPF adjacency not forming after VLAN change. Root cause: area mismatch"],
    metadatas=[{"resolution": "Corrected area on affected interface", "protocol": "ospf"}]
)

# ── Demo tools ────────────────────────────────────────────────
DEMO_TOOLS = {
    "show_ospf_neighbors": lambda: "R1 Gi0/1: neighbor 10.0.0.2 State INIT Dead 34",
    "show_ospf_interface_R1": lambda: "Area 0, Hello 10, Dead 40, MTU 1500",
    "show_ospf_interface_R2": lambda: "Area 1, Hello 10, Dead 40, MTU 1500",
}

print("LLM, ChromaDB, and demo tools ready.")

## Step 6 — The Four Nodes

Each node:
1. Receives the full current state
2. Does exactly one thing
3. Returns **only the fields that changed**

LangGraph merges the returned dict into the state. You never pass the whole state back — only the delta.

In [ ]:
def observe_node(state: AgentState) -> dict:
    """
    Node 1: OBSERVE
    Runs once at the start. Pulls everything the agent needs before reasoning.
    In production: also queries topology graph and device inventory.
    """
    query = state["user_query"]

    # Search ChromaDB for relevant past incidents
    rag_results = collection.query(query_texts=[query], n_results=3)
    rag_context = "\n".join(rag_results["documents"][0]) if rag_results["documents"][0] else "No prior incidents found."

    print(f"  [observe] RAG context: {rag_context[:80]}...")

    # Return only changed fields
    return {
        "phase":       "observe",
        "rag_context": rag_context,
        "evidence":    [],        # reset for this run
        "iteration":   0,
    }


def reason_node(state: AgentState) -> dict:
    """
    Node 2: REASON
    The core of the agent. Runs after every tool call.
    Asks the LLM: what do I know, what's my hypothesis, how confident am I, what next?
    """
    evidence_text = "\n".join(state["evidence"]) or "No evidence collected yet."

    prompt = f"""You are a network troubleshooting agent.
Query: {state['user_query']}
Background from knowledge base: {state['rag_context']}
Evidence collected so far:
{evidence_text}
Current hypothesis: {state['hypothesis'] or 'None yet'}

Respond in EXACTLY this format (no extra text):
THOUGHT: <your reasoning>
HYPOTHESIS: <current best explanation>
CONFIDENCE: <float 0.0-1.0>
NEXT_ACTION: <tool_name or DONE>
"""

    response = llm.invoke(prompt)

    # Parse the structured response
    lines = {}
    for line in response.content.strip().splitlines():
        if ": " in line:
            key, val = line.split(": ", 1)
            lines[key.strip()] = val.strip()

    confidence = float(lines.get("CONFIDENCE", 0.0))
    next_action = lines.get("NEXT_ACTION", "DONE")

    print(f"  [reason] iter={state['iteration']+1} | confidence={confidence:.2f} | next={next_action}")
    print(f"  [reason] hypothesis: {lines.get('HYPOTHESIS', '')[:80]}")

    return {
        "phase":        "reason",
        "hypothesis":   lines.get("HYPOTHESIS", state["hypothesis"]),
        "confidence":   confidence,
        "pending_tool": next_action,
        "iteration":    state["iteration"] + 1,
    }


def act_node(state: AgentState) -> dict:
    """
    Node 3: ACT
    Executes the tool that reason_node selected.
    Appends the result to the evidence list.
    Does NOT call the LLM.
    """
    tool_name = state["pending_tool"]

    # Look up and call the tool
    if tool_name in DEMO_TOOLS:
        result = DEMO_TOOLS[tool_name]()
    else:
        result = f"Tool '{tool_name}' not found in registry."

    print(f"  [act] ran tool: {tool_name}")
    print(f"  [act] result:   {result}")

    # Return only the new evidence and updated tool_calls log
    return {
        "phase":      "act",
        "evidence":   state["evidence"] + [f"[{tool_name}]: {result}"],
        "tool_calls": state["tool_calls"] + [tool_name],
    }


def verify_node(state: AgentState) -> dict:
    """
    Node 4: VERIFY
    Final node. Runs when confidence >= threshold or max_iterations is hit.
    Produces a clean, structured answer for the NOC engineer.
    """
    evidence_text = "\n".join(state["evidence"])

    prompt = f"""You are finalizing a network troubleshooting report.
Query: {state['user_query']}
Hypothesis: {state['hypothesis']}
Evidence:\n{evidence_text}
Confidence: {state['confidence']}

Write a final answer with:
1. Root cause (one sentence)
2. Evidence summary (bullet points)
3. Recommended action
4. Caveats (what you could not verify)
"""

    response = llm.invoke(prompt)

    print(f"  [verify] Final answer ready. confidence={state['confidence']:.2f}")

    return {
        "phase":        "verify",
        "final_answer": response.content.strip(),
        "done":         True,
    }


print("All four nodes defined.")

## Step 7 — The Routing Function

This is the intelligence of the graph. After every `reason_node` call, LangGraph asks this function: **where next?**

The function returns a string — the name of the next node. That is the entire branching logic of your agent. 12 lines of Python. Readable and testable.

In [ ]:
CONFIDENCE_THRESHOLD = 0.85  # Below this → collect more evidence. At/above → verify.


def route_after_reason(state: AgentState) -> str:
    """
    Called by LangGraph after every reason_node execution.
    Returns: 'verify' or 'act'
    
    Four conditions send the agent to verify:
    1. done flag is set
    2. confidence reached the threshold
    3. max iterations exhausted
    4. LLM said DONE explicitly
    """
    if state.get("done"):
        return "verify"

    if state["confidence"] >= CONFIDENCE_THRESHOLD:
        print(f"  [route] confidence {state['confidence']:.2f} >= {CONFIDENCE_THRESHOLD} → verify")
        return "verify"

    if state["iteration"] >= state["max_iterations"]:
        print(f"  [route] max iterations {state['max_iterations']} reached → verify")
        return "verify"

    if state["pending_tool"] == "DONE":
        print("  [route] LLM said DONE → verify")
        return "verify"

    # Still below threshold — collect more evidence
    return "act"


# Test the routing logic without running the graph
print("Routing test cases:")
test_states = [
    {"done": False, "confidence": 0.6, "iteration": 2, "max_iterations": 6, "pending_tool": "show_ospf"},
    {"done": False, "confidence": 0.9, "iteration": 3, "max_iterations": 6, "pending_tool": "show_ospf"},
    {"done": False, "confidence": 0.5, "iteration": 6, "max_iterations": 6, "pending_tool": "show_ospf"},
    {"done": False, "confidence": 0.5, "iteration": 2, "max_iterations": 6, "pending_tool": "DONE"},
]
for s in test_states:
    route = route_after_reason(s)
    print(f"  conf={s['confidence']}, iter={s['iteration']}/{s['max_iterations']}, tool={s['pending_tool']} → {route}")

## Step 8 — Build and Run the Graph

Now we wire everything together. `StateGraph` takes your TypedDict type, registers nodes, adds edges (fixed and conditional), and compiles.

**Fixed edges:** observe → reason, act → reason, verify → END  
**Conditional edge:** reason → (act or verify) based on `route_after_reason()`

In [ ]:
from langgraph.graph import StateGraph, END


def build_agent():
    graph = StateGraph(AgentState)

    # Register nodes
    graph.add_node("observe", observe_node)
    graph.add_node("reason",  reason_node)
    graph.add_node("act",     act_node)
    graph.add_node("verify",  verify_node)

    # Entry point — first node to execute
    graph.set_entry_point("observe")

    # Fixed edges — these always happen, no conditions
    graph.add_edge("observe", "reason")   # after observing, always reason
    graph.add_edge("act",     "reason")   # after acting, always reason again
    graph.add_edge("verify",  END)        # after verify, always end

    # Conditional edge — after reason, routing function decides where to go
    graph.add_conditional_edges(
        "reason",             # source node
        route_after_reason,   # routing function
        # map: function return value → destination node name
        {"act": "act", "verify": "verify"}
    )

    return graph.compile()


print("Graph compiled. Starting agent run...\n")
print("=" * 60)

agent = build_agent()

# Initial state — all fields at zero values
initial_state: AgentState = {
    "user_query":     "OSPF adjacency keeps flapping between R1 and R2. Neighbor is stuck in INIT state.",
    "phase":          "init",
    "iteration":      0,
    "max_iterations": 5,
    "rag_context":    "",
    "evidence":       [],
    "hypothesis":     "",
    "confidence":     0.0,
    "tool_calls":     [],
    "pending_tool":   None,
    "final_answer":   "",
    "done":           False,
}

result = agent.invoke(initial_state)

print("=" * 60)
print("\n=== FINAL ANSWER ===")
print(result["final_answer"])
print(f"\nConfidence:  {result['confidence']:.2f}")
print(f"Iterations:  {result['iteration']}")
print(f"Tools used:  {', '.join(result['tool_calls']) or 'none'}")

## What's Next

Your agent now has a proper state machine. In **Module 6** you add the Context Layer: an MCP server for network standards and a Neo4j knowledge graph for topology — so the agent knows YOUR network, not just networking in general.

---

**Module Challenge:** Run this agent with the OSPF flapping query. Post in **#agent-builds**: (1) how many iterations to reach confidence threshold, (2) tools called in order, (3) final confidence score.